In [9]:
import numpy as np
import xgboost as xgb
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

In [10]:
df_improvement = pd.read_csv("dataset/df_clean.csv")

# Although XGBoost doesn't need scaled inputs, this is done to ensure fairness for comparison between baseline and improved model
df_improvement["Minutes of Use"] = df_improvement["Seconds of Use"] / 60
df_improvement["Charge Amount/10"] = df_improvement["Charge  Amount"] / 10
df_improvement = df_improvement.drop(columns=["Seconds of Use", "Charge  Amount"])

print(f"Shape: {df_improvement.shape}")
df_train, df_test = train_test_split(df_improvement, test_size=0.4, stratify=df_improvement['Status'], random_state=42)
print(f"Shape for training set: {df_train.shape}")
print(f"Shape for test set: {df_test.shape}")

Shape: (2850, 14)
Shape for training set: (1710, 14)
Shape for test set: (1140, 14)


In [ ]:
# Drop the targets for X so that it prevents data leakage 
X_train = df_train.drop(columns=["Subscription  Length", "Churn"])
dtrain = xgb.DMatrix(X_train)

# Setting up the boundaries
# We know that no customers churned before the end of their Subscription Length (so it becomes the lower bound)
y_lower_bound = df_train['Subscription  Length'].values
# Where function is like the if() function, params are (condition, result if positive, result if negative)
y_upper_bound = np.where(df_train['Churn'] == 0, np.inf, df_train['Subscription  Length'].values)

dtrain.set_float_info('label_lower_bound', y_lower_bound)
dtrain.set_float_info('label_upper_bound', y_upper_bound)

X_test = df_test.drop(columns=["Subscription  Length", "Churn"])
dtest = xgb.DMatrix(X_test)

y_lower_test = df_test['Subscription  Length'].values
y_upper_test = np.where(df_test['Churn'] == 0, np.inf, df_test['Subscription  Length'].values)

dtest.set_float_info('label_lower_bound', y_lower_test)
dtest.set_float_info('label_upper_bound', y_upper_test)